# KoChatGPT 업그레이드: 데이터 정제·생성 전략·Reward Model 비교

본 노트북은 로컬 GPU 환경에서 KoGPT-2 베이스라인, 원본·정제 SFT 모델, Reward Model을 단계적으로 비교한 실험 기록이다. 데이터 품질 점검·정제, 생성 전략 탐색, RM reranking을 개선 전략으로 적용하고 결과를 정량·정성적으로 분석했다.

- 개선 전략: 기존 SFT 말뭉치의 구조·중복·길이·결측·문자 품질을 점검하고 보수적으로 정제
- 생성 전략: greedy decoding, beam search, top-k/top-p sampling의 정량·정성 비교
- 비교 대상: KoGPT-2 baseline → 정제 데이터 SFT 모델
- 정량 평가: 고정 평가 세트의 ROUGE-L·BLEU (참조 응답이 있는 held-out SFT 데이터)
- 정성 평가: 동일 프롬프트에서의 지시 이행·관련성·반복·문장 완결성 비교
- 확장 분석: Reward Model의 good/bad/worst 선호 순위 일치 여부


## 실험 환경 및 재현 조건

- 로컬 Windows GPU 환경에서 Jupyter Notebook을 실행했다. 패키지 관리는 `uv` 기반 가상환경을 사용했다.
- 노트북과 같은 경로에 `data_kochatgpt` 폴더를 배치해 SFT·RM·PPO 데이터를 참조했다. 제공 파일은 확장자와 달리 JSON 배열 형식이므로 전용 로더로 읽었다.
- GPU 메모리 제약에 맞춰 batch size, 최대 시퀀스 길이, gradient accumulation을 조정할 수 있도록 학습 설정을 분리했다.



In [ ]:
%pip install -q torch transformers datasets accelerate evaluate rouge_score sacrebleu pandas matplotlib
# Restart the kernel once if this cell installs packages for the first time.

In [ ]:
from pathlib import Path
import json
import random
import re

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, PreTrainedTokenizerFast, DataCollatorForSeq2Seq,
    Trainer, TrainingArguments, set_seed
)

set_seed(42)
candidate_roots = [Path.cwd(), Path.cwd() / 'kochatgpt-upgrade', Path.cwd() / 'sandbox' / 'kochatgpt-upgrade']
PROJECT_ROOT = next((path for path in candidate_roots if (path / 'data_kochatgpt').exists()), Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data_kochatgpt'
OUTPUT_DIR = PROJECT_ROOT / 'artifacts'
OUTPUT_DIR.mkdir(exist_ok=True)
BASE_MODEL_ID = 'skt/kogpt2-base-v2'
SFT_PATH = DATA_DIR / 'kochatgpt_1_SFT.jsonl'
RM_PATH = DATA_DIR / 'kochatgpt_2_RM.jsonl'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'{DEVICE=}, {PROJECT_ROOT=}')
assert SFT_PATH.exists(), f'SFT data not found: {SFT_PATH}. Open Jupyter from the project folder or check DATA_DIR.'

## 1. 데이터 EDA와 정제

확인 결과 SFT 파일은 12,000개 레코드의 JSON 배열이며, 각 레코드는 `prompt`, `completion`, `tokens` 키를 가집니다. 정제는 결측·중복·제어문자·비정상 길이를 대상으로 한 보수적 필터링으로 한정합니다. 제거 수와 표본은 결과 분석에 기록합니다.

> **augmentation 범위**: 이번 실험은 기본 정제 자체의 효과를 분리해 확인하기 위해 augmentation을 의도적으로 적용하지 않는다. 원본 데이터에 사실 오류와 이상 샘플이 포함돼 있어, 검증 없는 증강은 노이즈를 확대할 가능성이 있기 때문이다.


In [ ]:
# Load both standard JSONL files and the provided JSON-array files.
def read_records(path: Path):
    raw_text = path.read_text(encoding='utf-8-sig').strip()
    # The provided files use a JSON array despite their .jsonl extension.
    if raw_text.startswith('['):
        records = json.loads(raw_text)
    else:
        records = [json.loads(line) for line in raw_text.splitlines() if line.strip()]
    if not isinstance(records, list):
        raise ValueError(f'Expected a list of records: {path}')
    return records

# Normalize alternate source field names into one prompt/completion schema.
def normalize_sft_record(record):
    prompt = record.get('prompt') or record.get('instruction') or record.get('input') or ''
    completion = record.get('completion') or record.get('output') or record.get('response') or ''
    return {'prompt': str(prompt).strip(), 'completion': str(completion).strip()}

# Normalize whitespace before length filtering and duplicate detection.
def clean_text(text: str):
    text = re.sub(r'\s+', ' ', text).strip()
    return text

raw_sft = [normalize_sft_record(row) for row in read_records(SFT_PATH)]
clean_sft = []
for row in raw_sft:
    prompt, completion = clean_text(row['prompt']), clean_text(row['completion'])
    has_control_char = any(ord(char) < 32 and char not in '\n\t' for char in prompt + completion)
    if 5 <= len(prompt) <= 512 and 15 <= len(completion) <= 1200 and not has_control_char:
        clean_sft.append({'prompt': prompt, 'completion': completion})

before = pd.DataFrame(raw_sft)
after = pd.DataFrame(clean_sft).drop_duplicates().reset_index(drop=True)
print(f'raw={len(before):,}, cleaned={len(after):,}, removed={len(before)-len(after):,}')
print('Schema:', list(read_records(SFT_PATH)[0].keys()))
display(after.assign(prompt_len=after.prompt.str.len(), completion_len=after.completion.str.len())
        [['prompt_len', 'completion_len']].describe())
display(after.sample(min(3, len(after)), random_state=42))
after.to_json(OUTPUT_DIR / 'sft-cleaned.jsonl', orient='records', lines=True, force_ascii=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
after.prompt.str.len().plot.hist(ax=axes[0], bins=30, title='Prompt length')
after.completion.str.len().plot.hist(ax=axes[1], bins=30, title='Completion length')
plt.tight_layout()
# Record the observed distributions in the final discussion.

## 2. 공통 프롬프트와 베이스라인

SFT와 평가는 같은 프롬프트 템플릿을 사용해 비교 조건을 통제했다. 학습 전 KoGPT-2의 생성 결과를 저장해 베이스라인 비교 기준으로 활용했다.


In [ ]:
PROMPT_TEMPLATE = '### Instruction:\n{prompt}\n\n### Response:\n'
# Use the KoGPT-2 tokenizer explicitly to keep token IDs aligned with the model vocabulary.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    BASE_MODEL_ID,
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>',
)
# Keep an untouched KoGPT-2 copy as the pre-training baseline.
baseline_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID).to(DEVICE)
baseline_model.eval()
assert tokenizer.vocab_size == baseline_model.config.vocab_size, (
    f'Tokenizer vocab={tokenizer.vocab_size}, model vocab={baseline_model.config.vocab_size}'
)
print(f'KoGPT-2 tokenizer verified: vocab_size={tokenizer.vocab_size}')

# Generate one response with fixed controls for reproducible model comparisons.
def generate_answer(model, prompt, max_new_tokens=96, temperature=0.7, top_p=0.9, top_k=50, seed=None):
    if seed is not None:
        set_seed(seed)
    text = PROMPT_TEMPLATE.format(prompt=prompt)
    encoded = tokenizer(text, return_tensors='pt').to(DEVICE)
    with torch.inference_mode():
        output_ids = model.generate(
            **encoded, max_new_tokens=max_new_tokens, do_sample=True,
            temperature=temperature, top_p=top_p, top_k=top_k,
            repetition_penalty=1.15, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded[len(text):].strip()

evaluation_indices = after.sample(min(20, len(after)), random_state=42).index
evaluation_set = after.loc[evaluation_indices].reset_index(drop=True)
evaluation_set['baseline'] = [generate_answer(baseline_model, p, seed=42 + i)
                              for i, p in enumerate(evaluation_set.prompt)]
display(evaluation_set[['prompt', 'completion', 'baseline']].head(5))
evaluation_set.to_json(OUTPUT_DIR / 'baseline-generations.jsonl', orient='records', lines=True, force_ascii=False)

## 3. SFT 데이터셋과 학습

Prompt 부분의 label을 `-100`으로 설정해 loss에서 제외했다. 따라서 모델은 지시문을 그대로 복사하는 대신 completion 토큰을 예측하도록 학습했다.


In [ ]:
MAX_LENGTH = 256
TRAIN_BATCH_SIZE = 1
NUM_EPOCHS = 1

# Exclude the fixed evaluation prompts from SFT training to prevent leakage.
train_frame = after.drop(evaluation_indices).reset_index(drop=True)
train_dataset = Dataset.from_pandas(train_frame, preserve_index=False)

# Concatenate instruction and response, then create causal-LM training labels.
def tokenize_sft(example):
    source = PROMPT_TEMPLATE.format(prompt=example['prompt'])
    target = example['completion'] + tokenizer.eos_token
    source_ids = tokenizer(source, add_special_tokens=False)['input_ids']
    target_ids = tokenizer(target, add_special_tokens=False)['input_ids']
    input_ids = (source_ids + target_ids)[:MAX_LENGTH]
    # Ignore the instruction tokens so loss is computed only on the response tokens.
    labels = ([-100] * len(source_ids) + target_ids)[:MAX_LENGTH]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids), 'labels': labels}

tokenized_train = train_dataset.map(tokenize_sft, remove_columns=train_dataset.column_names)
# Initialize a fresh model copy so baseline weights are never updated.
sft_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID).to(DEVICE)
max_input_id = max(max(row['input_ids']) for row in tokenized_train)
assert max_input_id < sft_model.config.vocab_size, (
    f'Invalid token ID {max_input_id}; model vocab size is {sft_model.config.vocab_size}. '
    'Restart the kernel and run the tokenizer cell again.'
)
assert all(label == -100 or 0 <= label < sft_model.config.vocab_size
           for row in tokenized_train for label in row['labels'])
print(f'Training input IDs verified: max_id={max_input_id}')
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=sft_model, label_pad_token_id=-100, padding=True)
# Configure a resource-aware one-epoch SFT run.
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'sft-checkpoints'),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=8,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=2e-5,
    fp16=False,  # Enable only after the first stable full-precision run.
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
)
trainer = Trainer(model=sft_model, args=training_args, train_dataset=tokenized_train, data_collator=collator)
trainer.train()
trainer.save_model(str(OUTPUT_DIR / 'sft-model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'sft-model'))

## 4. 원본 데이터 SFT 대조군

정제 효과를 분리하기 위해 정제 전 원본 데이터에도 같은 KoGPT-2와 평가 프롬프트를 사용해 SFT를 수행했다. 정제 모델과 원본 데이터 모델의 업데이트 횟수를 모두 721 step으로 고정해 비교 조건을 통제했다.


In [ ]:
# Train a control model on unfiltered records using the same held-out prompts.
raw_frame = pd.DataFrame([
    {
        'prompt': clean_text(row.get('prompt', '')),
        'completion': clean_text(row.get('completion', '')),
    }
    for row in read_records(SFT_PATH)
])
evaluation_keys = set(zip(evaluation_set['prompt'], evaluation_set['completion']))
raw_train_frame = raw_frame.loc[
    ~raw_frame.apply(lambda row: (row['prompt'], row['completion']) in evaluation_keys, axis=1)
].reset_index(drop=True)
print(f'Raw control training records: {len(raw_train_frame):,}')

raw_train_dataset = Dataset.from_pandas(raw_train_frame, preserve_index=False)
tokenized_raw_train = raw_train_dataset.map(
    tokenize_sft, remove_columns=raw_train_dataset.column_names
)
raw_sft_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID).to(DEVICE)
raw_max_input_id = max(max(row['input_ids']) for row in tokenized_raw_train)
assert raw_max_input_id < raw_sft_model.config.vocab_size
raw_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=raw_sft_model, label_pad_token_id=-100, padding=True
)
raw_training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'raw-sft-checkpoints'),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=8,
    # Match the cleaned-data SFT update count for a controlled comparison.
    max_steps=721,
    learning_rate=2e-5,
    fp16=False,
    logging_steps=10,
    save_strategy='steps',
    save_steps=721,
    report_to='none',
)
raw_trainer = Trainer(
    model=raw_sft_model, args=raw_training_args,
    train_dataset=tokenized_raw_train, data_collator=raw_collator
)
raw_trainer.train()
raw_trainer.save_model(str(OUTPUT_DIR / 'raw-sft-model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'raw-sft-model'))


## 5. 정량·정성 비교

BLEU와 ROUGE는 단일 참조 답변 기반 생성 평가의 한계가 있으므로, 동일 프롬프트의 실제 출력을 함께 해석했다. 점수는 절대적 품질이 아니라 **동일 조건에서의 상대 비교**에 활용했다.


In [ ]:
sft_model.eval()
raw_sft_model.eval()
evaluation_set['raw_sft'] = [generate_answer(raw_sft_model, p, seed=42 + i)
                             for i, p in enumerate(evaluation_set.prompt)]
evaluation_set['sft'] = [generate_answer(sft_model, p, seed=42 + i)
                         for i, p in enumerate(evaluation_set.prompt)]

import evaluate
rouge = evaluate.load('rouge')
bleu = evaluate.load('sacrebleu')

# Compute reference-overlap metrics only for relative model comparison.
def calculate_metrics(predictions, references):
    rouge_score = rouge.compute(predictions=list(predictions), references=list(references))['rougeL']
    bleu_score = bleu.compute(predictions=list(predictions), references=[[r] for r in references])['score']
    return {'ROUGE-L': round(rouge_score, 4), 'BLEU': round(bleu_score, 2)}

metric_table = pd.DataFrame([
    {'model': 'KoGPT-2 baseline', **calculate_metrics(evaluation_set.baseline, evaluation_set.completion)},
    {'model': 'Raw-data SFT (721 steps)', **calculate_metrics(evaluation_set.raw_sft, evaluation_set.completion)},
    {'model': 'Cleaned-data SFT (721 steps)', **calculate_metrics(evaluation_set.sft, evaluation_set.completion)},
])
display(metric_table)
display(evaluation_set[['prompt', 'completion', 'baseline', 'raw_sft', 'sft']].head(10))
metric_table.to_csv(OUTPUT_DIR / 'comparison-metrics.csv', index=False, encoding='utf-8-sig')
evaluation_set.to_json(OUTPUT_DIR / 'comparison-generations.jsonl', orient='records', lines=True, force_ascii=False)

### 5.1 생성 전략 비교 실험

프로젝트 요구사항의 generation 기법 실험을 위해 정제 SFT 모델에 greedy decoding, beam search, top-k/top-p sampling을 적용했다. 모델·평가 프롬프트·최대 생성 길이는 고정하고 decoding 설정만 변경했다. 20개 hold-out 평가셋은 작으므로 지표 차이를 탐색적 결과로 해석하고, 정량 지표와 동일 프롬프트의 출력 사례를 함께 확인했다.


In [ ]:
DECODING_CONFIGS = [
    {'name': 'Greedy', 'do_sample': False, 'num_beams': 1},
    {'name': 'Beam search (4)', 'do_sample': False, 'num_beams': 4},
    {
        'name': 'Top-k / Top-p sampling', 'do_sample': True,
        'temperature': 0.7, 'top_k': 50, 'top_p': 0.9,
    },
]

# Vary only decoding parameters while keeping model and prompts fixed.
def generate_with_config(model, prompt, config, seed):
    set_seed(seed)
    prefix = PROMPT_TEMPLATE.format(prompt=prompt)
    encoded = tokenizer(prefix, return_tensors='pt').to(DEVICE)
    generation_kwargs = {
        'max_new_tokens': 96,
        'repetition_penalty': 1.15,
        'no_repeat_ngram_size': 3,
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
    }
    # Keep model, prompts, and length fixed; vary only decoding parameters.
    generation_kwargs.update({key: value for key, value in config.items() if key != 'name'})
    with torch.inference_mode():
        output_ids = model.generate(**encoded, **generation_kwargs)
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded[len(prefix):].strip()

if 'reward_model' in globals():
    reward_model.to('cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
sft_model.to(DEVICE)
sft_model.eval()
decoding_rows = []
for config_index, config in enumerate(DECODING_CONFIGS):
    predictions = [
        generate_with_config(sft_model, prompt, config, seed=20_000 + config_index * 100 + prompt_index)
        for prompt_index, prompt in enumerate(evaluation_set['prompt'])
    ]
    decoding_rows.extend([
        {'strategy': config['name'], 'prompt': prompt, 'reference': reference, 'prediction': prediction}
        for prompt, reference, prediction in zip(evaluation_set['prompt'], evaluation_set['completion'], predictions)
    ])

decoding_results = pd.DataFrame(decoding_rows)
decoding_metric_rows = []
for strategy, group in decoding_results.groupby('strategy', sort=False):
    decoding_metric_rows.append({'strategy': strategy, **calculate_metrics(group['prediction'], group['reference'])})
decoding_metric_table = pd.DataFrame(decoding_metric_rows)
display(decoding_metric_table)
display(decoding_results.pivot(index='prompt', columns='strategy', values='prediction').head(5))
decoding_metric_table.to_csv(OUTPUT_DIR / 'decoding-strategy-metrics.csv', index=False, encoding='utf-8-sig')
decoding_results.to_json(OUTPUT_DIR / 'decoding-strategy-results.jsonl', orient='records', lines=True, force_ascii=False)


## 6. SFT 결과 분석

### 실험 결과 요약

- 데이터: 원본 SFT 12,000건을 보수적으로 정제하여 11,542건을 사용했다. 길이·결측·제어문자 기준으로 458건이 제외되었다.
- 학습: 정제 SFT와 원본 SFT를 모두 KoGPT-2에서 721 step 학습해 업데이트 횟수를 맞췄다. 정제 SFT의 training loss는 step 10에서 3.655, 마지막 기록(step 720)에서 2.714였다.
- 정량 평가: 고정 hold-out 20개 프롬프트에서 baseline의 ROUGE-L/BLEU는 0.0000/0.17, 원본 SFT는 0.1944/0.59, 정제 SFT는 0.1911/0.91이었다.
- 생성 전략: 정제 SFT에 beam search(4)를 적용하면 ROUGE-L 0.1950 / BLEU 2.49로, 기본 sampling(0.1911/0.91)과 원본 SFT(0.1944/0.59) 모두를 상회했다.

### 정성 비교

Baseline KoGPT-2는 질문과 무관한 해시태그·SNS 문장·영문 조각을 주로 생성했다. 반면 원본·정제 SFT 모델은 대부분 AI 어시스턴트 응답 형식을 유지하고 질문의 주제를 일부 반영했다. 따라서 SFT는 지시문 형식과 한국어 응답 스타일을 학습하는 데 효과가 있었다.

정제 SFT는 BLEU가 원본 SFT보다 높았지만(0.91 vs. 0.59), ROUGE-L은 근소하게 낮았다(0.1911 vs. 0.1944). 따라서 정제 단독으로는 모든 자동 지표를 일관되게 높였다고 단정할 수 없다. 생성 사례에서도 두 SFT 모델은 모두 사실형 질문에서 환각·오답을 만들거나 답변하기 어렵습니다 같은 회피형 표현을 반복했다. 정제 모델은 일부 응답이 더 길고 문장 형식이 갖춰졌으나, 그만큼 근거 없는 세부 정보를 덧붙이는 경우도 있었다.

### 생성 전략 비교

정제 SFT 모델에 greedy, beam search(4), top-k/top-p sampling을 적용한 결과, beam search(4)가 ROUGE-L 0.1950 / BLEU 2.49로 가장 높았다. Greedy는 0.1532/1.47, sampling은 0.0643/1.45를 기록했다. Beam search는 높은 확률의 토큰을 안정적으로 선택하므로 지표가 높았지만, 답변하기 어렵습니다와 같은 회피형 표현도 함께 강화했다(출력 확인 기준 20개 중 13개). Sampling은 회피 비율이 상대적으로 낮았지만(9개) 문장 안정성과 지표 모두 하락했다. Decoding 전략은 출력 분포를 조절할 수 있으나, 학습 데이터의 회피형 응답 편향이나 KoGPT-2의 사실성 한계를 근본적으로 해결하지는 못한다.

정제 데이터 + beam search 조합(ROUGE-L 0.1950, BLEU 2.49)은 원본 데이터 SFT + sampling(0.1944/0.59)을 두 지표 모두에서 상회했다. 데이터 정제 단독으로는 ROUGE-L 향상이 확인되지 않았지만, 생성 전략 변경을 함께 적용하면 정량적 향상이 나타났다.

### 한계

평가 데이터 중 일부에는 다른 레코드의 직렬화 조각이 prompt에 섞인 사례가 있었고, 원본 completion 자체에도 모호하거나 부정확한 답변이 포함된다. 20개 평가셋만으로는 지표 차이를 확정하기 어렵기 때문에, 보다 신뢰할 수 있는 비교를 위해서는 더 큰 고정 평가셋과 반복 생성 평균이 필요하다. 또한 직렬화 조각·언어 혼입·사실 오류를 제거하는 내용 기반 정제와, 사람이 표본을 점검하는 품질 검수가 선행되어야 한다.

## 7. Reward Model 학습 및 분석

RM 데이터는 10,220개 prompt마다 세 개의 completion과 선호 순위 `ranking`을 제공한다. 각 레코드에서 최상위 응답과 하위 두 응답을 비교하는 선호 쌍을 만들고, SFT 모델을 초기값으로 한 GPT-2 sequence classification 모델이 `chosen_reward > rejected_reward`가 되도록 학습했다.

> **파일명 확인**: 프로젝트 안내의 `kochatgpt_1_RM.jsonl` 표기와 달리, 실제 제공 데이터 파일은 `kochatgpt_2_RM.jsonl`이다. 본 실험은 실제 파일명을 기준으로 사용했다.

로컬 실행 시간과 자원 제약을 고려해 학습 선호 쌍은 3,000개로 제한했다. 평가는 학습에 포함되지 않은 prompt를 사용해 데이터 누수를 방지했다.


In [ ]:
from torch.utils.data import DataLoader, Dataset as TorchDataset
from torch.optim import AdamW
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification

RM_MAX_TRAIN_PAIRS = 3000
RM_MAX_EVAL_RECORDS = 500
RM_BATCH_SIZE = 2
RM_NUM_EPOCHS = 1
RM_MAX_LENGTH = 256

# Convert each ranked response triplet into preferred-versus-lower pairs.
def build_preference_pairs(records):
    pairs = []
    for record in records:
        prompt = clean_text(str(record.get('prompt', '')))
        ranking = record.get('ranking', [])
        if not prompt or len(ranking) < 2:
            continue
        # The first index is preferred; compare it with each lower-ranked response.
        best_index = int(ranking[0])
        best = clean_text(str(record.get(f'completion_{best_index}', '')))
        for lower_index in ranking[1:]:
            rejected = clean_text(str(record.get(f'completion_{int(lower_index)}', '')))
            if best and rejected:
                pairs.append({'prompt': prompt, 'chosen': best, 'rejected': rejected})
    return pairs

rm_records = read_records(RM_PATH)
rm_rng = random.Random(42)
rm_rng.shuffle(rm_records)
rm_split = int(len(rm_records) * 0.9)
rm_train_records, rm_eval_records = rm_records[:rm_split], rm_records[rm_split:]
rm_train_pairs = build_preference_pairs(rm_train_records)
if RM_MAX_TRAIN_PAIRS is not None:
    rm_rng.shuffle(rm_train_pairs)
    rm_train_pairs = rm_train_pairs[:RM_MAX_TRAIN_PAIRS]
if RM_MAX_EVAL_RECORDS is not None:
    rm_eval_records = rm_eval_records[:RM_MAX_EVAL_RECORDS]
print(f'RM records: train={len(rm_train_records):,}, eval={len(rm_eval_records):,}')
print(f'RM preference pairs used for training: {len(rm_train_pairs):,}')
display(pd.DataFrame(rm_train_pairs).head(3))


In [ ]:
# Wrap preference pairs in a PyTorch dataset for batched RM training.
class PreferencePairDataset(TorchDataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        return self.pairs[index]

# Tokenize chosen and rejected texts separately for reward-score comparison.
def rm_collate(batch):
    chosen_texts = [PROMPT_TEMPLATE.format(prompt=item['prompt']) + item['chosen'] for item in batch]
    rejected_texts = [PROMPT_TEMPLATE.format(prompt=item['prompt']) + item['rejected'] for item in batch]
    common = {'padding': True, 'truncation': True, 'max_length': RM_MAX_LENGTH, 'return_tensors': 'pt'}
    return {
        'chosen': tokenizer(chosen_texts, **common),
        'rejected': tokenizer(rejected_texts, **common),
    }

# Move no-longer-needed generation models off the GPU before RM training.
for model_name in ('baseline_model', 'raw_sft_model', 'sft_model'):
    if model_name in globals():
        globals()[model_name].to('cpu')
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Add a scalar reward head on top of the SFT backbone.
reward_model = AutoModelForSequenceClassification.from_pretrained(
    str(OUTPUT_DIR / 'sft-model'), num_labels=1, ignore_mismatched_sizes=True
)
reward_model.config.pad_token_id = tokenizer.pad_token_id
reward_model.to(DEVICE)

# Use dynamic padding to batch preference pairs efficiently.
rm_loader = DataLoader(
    PreferencePairDataset(rm_train_pairs), batch_size=RM_BATCH_SIZE,
    shuffle=True, collate_fn=rm_collate, pin_memory=(DEVICE == 'cuda')
)
rm_optimizer = AdamW(reward_model.parameters(), lr=1e-5)
rm_history = []
reward_model.train()
for epoch in range(RM_NUM_EPOCHS):
    running_loss = 0.0
    for step, batch in enumerate(rm_loader, start=1):
        chosen_inputs = {key: value.to(DEVICE) for key, value in batch['chosen'].items()}
        rejected_inputs = {key: value.to(DEVICE) for key, value in batch['rejected'].items()}
        chosen_reward = reward_model(**chosen_inputs).logits.squeeze(-1)
        rejected_reward = reward_model(**rejected_inputs).logits.squeeze(-1)
        # Pairwise loss widens the score gap between preferred and rejected responses.
        loss = -F.logsigmoid(chosen_reward - rejected_reward).mean()
        rm_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        rm_optimizer.step()
        running_loss += loss.item()
        if step % 50 == 0 or step == len(rm_loader):
            average_loss = running_loss / step
            print(f'Epoch {epoch + 1}/{RM_NUM_EPOCHS} | step {step}/{len(rm_loader)} | loss {average_loss:.4f}')
    rm_history.append({'epoch': epoch + 1, 'pairwise_loss': running_loss / len(rm_loader)})

reward_model.save_pretrained(OUTPUT_DIR / 'reward-model')
tokenizer.save_pretrained(OUTPUT_DIR / 'reward-model')
pd.DataFrame(rm_history).to_csv(OUTPUT_DIR / 'rm-training-history.csv', index=False)
display(pd.DataFrame(rm_history))


In [ ]:
# Return one scalar reward for a prompt-response pair.
def reward_score(prompt, completion):
    text = PROMPT_TEMPLATE.format(prompt=clean_text(str(prompt))) + clean_text(str(completion))
    encoded = tokenizer(text, return_tensors='pt', truncation=True, max_length=RM_MAX_LENGTH).to(DEVICE)
    with torch.inference_mode():
        return float(reward_model(**encoded).logits.squeeze().item())

reward_model.eval()
rm_result_rows = []
for record in rm_eval_records:
    ranking = [int(index) for index in record['ranking']]
    scores = {index: reward_score(record['prompt'], record[f'completion_{index}']) for index in range(3)}
    predicted_ranking = sorted(scores, key=scores.get, reverse=True)
    rm_result_rows.append({
        'prompt': record['prompt'],
        'expected_ranking': ranking,
        'predicted_ranking': predicted_ranking,
        'good_score': scores[ranking[0]],
        'bad_score': scores[ranking[1]],
        'worst_score': scores[ranking[2]],
        'top1_correct': predicted_ranking[0] == ranking[0],
        'strict_order_correct': predicted_ranking == ranking,
    })

rm_results = pd.DataFrame(rm_result_rows)
print(f"Top-1 preference accuracy: {rm_results['top1_correct'].mean():.3f}")
print(f"Strict ranking accuracy: {rm_results['strict_order_correct'].mean():.3f}")
display(rm_results[['prompt', 'good_score', 'bad_score', 'worst_score', 'predicted_ranking', 'strict_order_correct']].head(10))
rm_results.to_json(OUTPUT_DIR / 'rm-evaluation.jsonl', orient='records', lines=True, force_ascii=False)


### 7.1 RM reranking: SFT 후보 선택 실험

PPO 없이도 RM을 활용할 수 있는지 확인했다. 동일한 평가 프롬프트마다 SFT 모델이 4개의 후보 응답을 생성하고, RM이 가장 높은 reward를 받은 후보를 선택했다. 첫 번째 후보만 사용하는 SFT 단일 생성과 RM이 선택한 응답을 ROUGE-L·BLEU 및 출력 사례로 비교했다. 후보 생성 수가 4배가 되므로, 이 실험은 생성 품질과 추론 비용 사이의 trade-off도 함께 보여준다.


In [ ]:
RERANK_CANDIDATE_COUNT = 4
RERANK_MAX_NEW_TOKENS = 96

# Reload the SFT generator because it was moved to CPU before RM training.
rerank_generator = AutoModelForCausalLM.from_pretrained(OUTPUT_DIR / 'sft-model').to(DEVICE)
rerank_generator.eval()
reward_model.to(DEVICE)
reward_model.eval()

# Compare the first SFT candidate with the highest-reward candidate in its pool.
rerank_rows = []
for prompt_index, prompt in enumerate(evaluation_set['prompt']):
    candidates = [
        generate_answer(
            rerank_generator, prompt, max_new_tokens=RERANK_MAX_NEW_TOKENS,
            seed=10_000 + prompt_index * RERANK_CANDIDATE_COUNT + candidate_index
        )
        for candidate_index in range(RERANK_CANDIDATE_COUNT)
    ]
    rewards = [reward_score(prompt, candidate) for candidate in candidates]
    # RM reranking selects the highest-reward candidate without updating the SFT generator.
    selected_index = int(np.argmax(rewards))
    row = {
        'prompt': prompt,
        'reference': evaluation_set.loc[prompt_index, 'completion'],
        'single_sft': candidates[0],
        'single_sft_reward': rewards[0],
        'reranked_sft': candidates[selected_index],
        'reranked_reward': rewards[selected_index],
        'selected_index': selected_index,
    }
    for candidate_index, (candidate, reward) in enumerate(zip(candidates, rewards)):
        row[f'candidate_{candidate_index}'] = candidate
        row[f'reward_{candidate_index}'] = reward
    rerank_rows.append(row)

rerank_results = pd.DataFrame(rerank_rows)
rerank_metric_table = pd.DataFrame([
    {'model': 'SFT single candidate', **calculate_metrics(rerank_results['single_sft'], rerank_results['reference'])},
    {'model': 'SFT + RM reranking (best of 4)', **calculate_metrics(rerank_results['reranked_sft'], rerank_results['reference'])},
])
display(rerank_metric_table)
display(rerank_results[['prompt', 'single_sft_reward', 'reranked_reward', 'selected_index', 'single_sft', 'reranked_sft']].head(10))
rerank_metric_table.to_csv(OUTPUT_DIR / 'rm-reranking-metrics.csv', index=False, encoding='utf-8-sig')
rerank_results.to_json(OUTPUT_DIR / 'rm-reranking-results.jsonl', orient='records', lines=True, force_ascii=False)


### 7.2 RM 결과 해석

RM은 held-out 선호 순위에서 Top-1 accuracy 90.8%, strict ranking accuracy 82.2%를 기록해 학습 데이터와 같은 형식의 선호 신호를 대체로 재현했다. 그러나 best-of-4 reranking에서는 첫 후보보다 평균 reward가 5.28 높아졌음에도 ROUGE-L·BLEU가 각각 0.2256/2.27에서 0.1583/1.00으로 하락했다.

이는 RM reward를 최적화하는 것과 참조 답변에 가까운 응답을 고르는 것이 다를 수 있음을 뜻한다. 실제 후보를 확인하면 `AI 어시스턴트`, `정확한 정보를 제공하기 어렵습니다` 같은 회피형 표현이 상대적으로 높은 reward를 받는 경향이 보인다. 따라서 RM은 선호 순위 판별기이자 reranking 도구로 구현됐지만, 현재 보상 체계만으로 최종 생성 품질이 개선됐다고 결론내릴 수는 없다.


## 8. 최종 결과 및 결론

### 8.1 실험 요약

본 프로젝트는 KoGPT-2를 대상으로 (1) 원본·정제 SFT 데이터를 통한 지도 미세조정, (2) 선호 쌍으로 학습한 Reward Model 구축을 수행했다. 정제 데이터는 원본 SFT 12,000건에서 결측·제어문자·비정상 길이 기준으로 458건을 제외한 11,542건이다. 원본·정제 SFT는 모두 721 step으로 학습해 업데이트 횟수를 통제했다. RM은 SFT 체크포인트를 초기값으로 사용하고, 학습용 선호 쌍 3,000개와 분리된 평가 prompt 500개를 사용했다.

### 8.2 정량 결과

| 구분 | ROUGE-L | BLEU | 해석 |
| --- | ---: | ---: | --- |
| KoGPT-2 baseline | 0.0000 | 0.17 | 질문과 무관한 SNS·해시태그 문장이 주로 생성됨 |
| 원본 데이터 SFT | **0.1944** | 0.59 | 참조 응답과의 순서·표현 겹침이 가장 높음 |
| 정제 데이터 SFT | 0.1911 | **0.91** | 일부 핵심 어휘의 일치가 더 높음 |

정제 SFT의 생성 전략 비교에서는 greedy가 ROUGE-L/BLEU 0.1532/1.47, beam search(4)가 **0.1950/2.49**, top-k/top-p sampling이 0.0643/1.45를 기록했다. 자동 지표만 기준으로 하면 beam search(4)가 가장 높았으므로 최종 decoding 후보로 선택할 수 있다. 결과적으로 정제 데이터 + beam search 조합(ROUGE-L 0.1950, BLEU 2.49)은 원본 데이터 SFT(0.1944/0.59)를 두 지표 모두에서 상회하여, 데이터 정제와 생성 전략 변경의 복합 효과가 정량적 향상으로 이어졌다.

RM의 1 epoch pairwise loss는 0.2528이었다. 보지 않은 500개 prompt에서 최선 응답을 가장 높은 보상으로 선택한 Top-1 preference accuracy는 **90.8%**, good > bad > worst 전체 순서를 맞힌 strict ranking accuracy는 **82.2%**였다. 평균 reward는 good 22.52, bad 2.88, worst -14.68로 선호도에 따른 점수 분리가 확인됐다.

RM reranking에서는 프롬프트당 SFT 후보 4개를 생성했다. 20개 중 17개에서 첫 후보가 아닌 다른 후보가 선택됐고, 선택된 응답의 reward는 첫 후보 대비 평균 **5.28** 상승했다. 그러나 참조 응답 기반 지표는 SFT 단일 후보의 ROUGE-L/BLEU **0.2256/2.27**에서 reranking 후 **0.1583/1.00**으로 하락했다.

### 8.3 결과 해석

SFT 적용 전 KoGPT-2는 질문과 무관한 SNS 문장과 해시태그를 생성했지만, SFT 후에는 대체로 질문을 인식한 한국어 답변 형식으로 전환됐다. 따라서 SFT는 지시 이행과 응답 형식 학습에 효과적이었다. 원본·정제 SFT의 자동 지표는 서로 다른 방향을 보였다. 정제 SFT의 BLEU는 높았지만 ROUGE-L은 원본 SFT보다 0.0033 낮았다. 따라서 이번 20개 hold-out 평가와 단일 생성 조건만으로 데이터 정제가 일관된 성능 향상을 만들었다고 단정할 수는 없다.

생성 전략 비교(Section 6)에서 beam search(4)가 가장 높은 지표를 보였지만 회피형 문구를 강화하는 경향도 함께 나타났다. Decoding 전략은 출력 분포를 조절할 수 있으나, 학습 데이터의 편향이나 KoGPT-2의 사실성 한계를 근본적으로 해결하지는 못한다.

RM은 분리된 평가 데이터에서 높은 선호 순위 정확도를 보였다. 다만 일부 역사·상식 질문에서는 good과 bad의 순서를 뒤바꿨고, 관계 조언처럼 응답의 질이 주관적인 질문에서는 worst와 bad의 구분에도 실패했다. RM의 높은 정확도는 원본 ranking이 모델별 생성 출처를 기준으로 자동 부여된 약한 선호 신호라는 점을 함께 고려해야 한다.

Reranking 결과는 RM의 reward를 높이는 것과 참조 답변에 가까운 응답을 고르는 것이 동일하지 않음을 보여준다. 실제로 RM은 `AI 어시스턴트`, `정확한 정보를 제공할 수 없습니다`와 같은 데이터 내 빈번한 회피형 문장을 높은 점수로 선택하는 경향을 보였다. 따라서 이번 RM은 후보의 선호 점수를 높이는 데에는 성공했지만, ROUGE-L·BLEU와 생성 사례 기준으로 최종 답변 품질을 개선했다고 볼 수는 없다.

### 8.4 결론과 한계

본 실험은 KoGPT-2에 SFT를 적용해 생성 결과를 개선하고, RM이 응답 선호도를 점수화할 수 있음을 확인했다. 또한 PPO 없이 RM reranking을 수행해 reward가 가장 높은 후보를 선택할 수 있음을 보였다. 그러나 reranking은 참조 기반 지표를 개선하지 못했고, RM이 회피형 문장에 과도한 보상을 주는 현상도 관찰됐다. SFT 모델에는 사실 오류·환각·회피형 문구 반복이 남아 있었고, 원본 데이터에도 직렬화 조각과 부정확한 답변이 포함돼 있었다.

### 8.5 회고 및 다음 실험

이번 실험에서 가장 크게 확인한 점은 모델 학습 단계보다 데이터와 평가 설계가 결과 해석을 좌우한다는 사실이다. 기본적인 길이·형식 정제와 decoding 변경만으로는 사실 오류, 언어 혼입, 직렬화 조각, 회피형 응답 같은 내용 품질 문제를 해결할 수 없었다. 또한 ROUGE-L과 BLEU가 서로 다른 방향을 보였기 때문에, 하나의 자동 지표만으로 생성 품질을 판단하는 것은 위험했다.

RM의 높은 ranking accuracy도 최종 사용자 품질 개선을 보장하지 않았다. RM은 학습 데이터의 선호 순위는 잘 재현했지만, reranking에서는 일반적이고 안전한 회피 문장을 높은 reward로 선택하는 경향을 보였다. 이 과정은 보상 함수를 최적화하는 것과 사람이 좋은 답변이라고 평가하는 것이 다를 수 있다는 RLHF의 핵심 난점을 보여준다.

다음 실험에서는 (1) 직렬화 조각·사실 오류·언어 혼입을 제거하는 내용 기반 정제, (2) 100개 이상의 고정 평가셋과 복수 seed 평균, (3) 사람이 직접 판단한 유용성·사실성·안전성 평가, (4) 사람 검수 선호 쌍으로 RM 재학습을 우선 수행해야 한다. 이 검증이 선행된 뒤에만 RM reranking을 확대하거나 PPO를 적용해 높은 reward의 응답을 직접 생성하도록 학습하는 것이 타당하다.

결론적으로, KoGPT-2 기반 SFT는 질문 응답 형식으로의 전환에는 성공했고 RM은 선호 순위 판별 능력을 보였다. 그러나 현재 데이터와 보상 체계만으로는 사실성 높은 대화 모델을 만들기 어렵다는 한계도 함께 확인했다. 이 한계를 정량·정성 결과로 제시한 것이 본 프로젝트의 핵심 성과다.
